# Summary

Explore the RAG evaluation datasets

In [1]:
import os, sys
import pandas as pd
import json
import time

# AWS Python
import boto3
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth


# Numantic utilities
utils_path = "/Users/stephengodfrey/Documents/Workbench/Numantic/utilities/.."
sys.path.insert(0, utils_path)
from utilities.osa_tools.authentication import ApiAuthentication

api_configs = ApiAuthentication(client="Numantic")


## Create a Bedrock Knowledge Base

In [2]:
"""
Complete Guide to Creating an Amazon Bedrock Knowledge Base
============================================================

Prerequisites:
1. S3 bucket with .txt files containing document text
2. AWS credentials configured
3. Python packages: boto3, opensearchpy

This script assumes .txt files are already uploaded to S3
"""


'\nComplete Guide to Creating an Amazon Bedrock Knowledge Base\n============================================================\n\nPrerequisites:\n1. S3 bucket with .txt files containing document text\n2. AWS credentials configured\n3. Python packages: boto3, opensearchpy\n\nThis script assumes .txt files are already uploaded to S3\n'

## Set up configuration and initialize

In [3]:

# ============================================================================
# SECTION 0: Configuration and Initialization
# ============================================================================
# --- Configuration ---
REGION_NAME = 'us-east-2'
BUCKET_NAME = 'rag-search-tests'
S3_PREFIX = 'documents/'
COLLECTION_NAME = 'rag-kb-collection'
KB_NAME = 'rag-search-kb'
INDEX_NAME = 'bedrock-knowledge-base-default-index'
ROLE_NAME = 'AmazonBedrockExecutionRoleForKnowledgeBase'
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'

# Use Admin for all Setup tasks to avoid AccessDenied
admin_session = boto3.Session(profile_name='ns-admin',
                              region_name=REGION_NAME)

aoss_client = admin_session.client('opensearchserverless', region_name=REGION_NAME)
iam_client = admin_session.client('iam')
sts_client = admin_session.client('sts')
bedrock_agent = admin_session.client('bedrock-agent', region_name=REGION_NAME)

account_id = sts_client.get_caller_identity()['Account']
current_user_arn = sts_client.get_caller_identity()['Arn']


## Create Security Policies

In [4]:

# --- Create OpenSearch Serverless Security Policies and Bedrock Role ---
def ensure_aoss_policy(name, policy_type, policy_doc):
    try:
        aoss_client.create_security_policy(name=name, type=policy_type, policy=json.dumps(policy_doc))
        print(f"Created {policy_type} policy: {name}")
    except Exception as e:
        if 'ConflictException' in str(e):
            print(f"{policy_type.capitalize()} policy already exists: {name}")
        else:
            raise


def ensure_execution_role():
    trust_pol = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock.amazonaws.com"},
                "Action": "sts:AssumeRole"
            }
        ]
    }

    try:
        role_arn = iam_client.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_pol)
        )['Role']['Arn']
        print(f"Created role: {ROLE_NAME}")
    except iam_client.exceptions.EntityAlreadyExistsException:
        role_arn = iam_client.get_role(RoleName=ROLE_NAME)['Role']['Arn']
        print(f"Role already exists: {ROLE_NAME}")

    exec_pol = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:ListBucket"],
                "Resource": [f"arn:aws:s3:::{BUCKET_NAME}", f"arn:aws:s3:::{BUCKET_NAME}/*"]
            },
            {
                "Effect": "Allow",
                "Action": ["aoss:APIAccessAll"],
                "Resource": ["*"]
            },
            {
                "Effect": "Allow",
                "Action": ["bedrock:InvokeModel"],
                "Resource": [f"arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}"]
            }
        ]
    }

    # Always upsert policy to avoid stale role permissions.
    iam_client.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName='BedrockPolicy',
        PolicyDocument=json.dumps(exec_pol)
    )
    print("Upserted inline role policy: BedrockPolicy")
    return role_arn


def ensure_data_access_policy(role_arn):
    policy_name = f"{COLLECTION_NAME}-acc"
    role_name = role_arn.split('/')[-1]

    access_policy = [{
        "Rules": [
            {
                "Resource": [f"collection/{COLLECTION_NAME}"],
                "Permission": ["aoss:*"],
                "ResourceType": "collection"
            },
            {
                "Resource": [f"index/{COLLECTION_NAME}/*"],
                "Permission": ["aoss:*"],
                "ResourceType": "index"
            }
        ],
        "Principal": [
            role_arn,
            f"arn:aws:sts::{account_id}:assumed-role/{role_name}/*",
            current_user_arn
        ]
    }]

    try:
        aoss_client.create_access_policy(
            name=policy_name,
            type='data',
            policy=json.dumps(access_policy)
        )
        print(f"Created data access policy: {policy_name}")
    except Exception as e:
        if 'ConflictException' in str(e):
            existing_policy = aoss_client.get_access_policy(name=policy_name, type='data')
            try:
                aoss_client.update_access_policy(
                    name=policy_name,
                    type='data',
                    policyVersion=existing_policy['accessPolicyDetail']['policyVersion'],
                    policy=json.dumps(access_policy)
                )
                print(f"Updated data access policy: {policy_name}")
            except Exception as ve:
                if 'No changes detected' in str(ve):
                    print(f"Data access policy already up to date: {policy_name}")
                else:
                    raise
        else:
            raise


ensure_aoss_policy(
    name=f'{COLLECTION_NAME}-enc',
    policy_type='encryption',
    policy_doc={
        "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]}],
        "AWSOwnedKey": True
    }
)

ensure_aoss_policy(
    name=f'{COLLECTION_NAME}-net',
    policy_type='network',
    policy_doc=[{
        "Rules": [
            {"ResourceType": "collection", "Resource": [f"collection/{COLLECTION_NAME}"]},
            {"ResourceType": "dashboard", "Resource": [f"collection/{COLLECTION_NAME}"]}
        ],
        "AllowFromPublic": True
    }]
)

role_arn = ensure_execution_role()
ensure_data_access_policy(role_arn)
print("✓ Security roles and policies established.")


Encryption policy already exists: rag-kb-collection-enc
Network policy already exists: rag-kb-collection-net
Role already exists: AmazonBedrockExecutionRoleForKnowledgeBase
Upserted inline role policy: BedrockPolicy
Data access policy already up to date: rag-kb-collection-acc
✓ Security roles and policies established.


## Check permissions

In [5]:
# Add before creating KB
print(f"Using role: {role_arn}")
try:
    role_name = role_arn.split('/')[-1]
    iam_client.get_role(RoleName=role_name)
    print("✓ Role exists")
except Exception as e:
    print(f"✗ Role issue: {e}")

Using role: arn:aws:iam::584560776394:role/AmazonBedrockExecutionRoleForKnowledgeBase
✓ Role exists


## Add security roles

In [6]:

# Wait for IAM and AOSS policy propagation before KB creation
MAX_WAIT_SECONDS = 180
SLEEP_SECONDS = 10

print(f"Using role: {role_arn}")
iam_client.get_role(RoleName=ROLE_NAME)

start = time.time()
while True:
    elapsed = time.time() - start
    if elapsed > MAX_WAIT_SECONDS:
        raise TimeoutError("Timed out waiting for IAM/AOSS policy propagation.")

    try:
        pol = aoss_client.get_access_policy(name=f"{COLLECTION_NAME}-acc", type='data')
        version = pol['accessPolicyDetail']['policyVersion']
        print(f"✓ Access policy visible (version {version}); proceeding.")
        break
    except Exception as e:
        print(f"Waiting for policy propagation ({int(elapsed)}s): {e}")
        time.sleep(SLEEP_SECONDS)


Using role: arn:aws:iam::584560776394:role/AmazonBedrockExecutionRoleForKnowledgeBase
✓ Access policy visible (version MTc3NjM4NDEwMzg1N18y); proceeding.


## Create vector index

In [7]:
# Set up OpenSearch client with AWS auth using admin_session
credentials = admin_session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, REGION_NAME, 'aoss')

# Get the collection endpoint
collection_endpoint = aoss_client.batch_get_collection(names=[COLLECTION_NAME])['collectionDetails'][0][
    'collectionEndpoint']
host = collection_endpoint.replace('https://', '')

os_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

# Create the index with proper mapping for Bedrock
index_body = {
    "settings": {
        "index.knn": True
    },
    "mappings": {
        "properties": {
            "bedrock-knowledge-base-default-vector": {
                "type": "knn_vector",
                "dimension": 1024,
                "method": {
                    "engine": "faiss",
                    "name": "hnsw"
                }
            },
            "AMAZON_BEDROCK_TEXT_CHUNK": {
                "type": "text"
            },
            "AMAZON_BEDROCK_METADATA": {
                "type": "text"
            }
        }
    }
}

# Create index if it doesn't exist
if not os_client.indices.exists(index=INDEX_NAME):
    os_client.indices.create(index=INDEX_NAME, body=index_body)
    print(f"✓ Vector index created: {INDEX_NAME}")
else:
    print(f"✓ Vector index already exists: {INDEX_NAME}. To delete, use os_client.indices.delete(index=INDEX_NAME)")
    # os_client.indices.delete(index=INDEX_NAME)

# Wait for index to be ready
time.sleep(5)

✓ Vector index created: bedrock-knowledge-base-default-index


## Set up Knowledge Base

In [10]:

### Delete knowledge base if it already exists

print(f"Checking for existing Knowledge Base named '{KB_NAME}'...")
existing_kb_id = None
paginator = bedrock_agent.get_paginator('list_knowledge_bases')
for page in paginator.paginate():
    for kb in page['knowledgeBaseSummaries']:
        if kb['name'] == KB_NAME:
            existing_kb_id = kb['knowledgeBaseId']
            print(f"Found existing KB {existing_kb_id}. Deleting...")
            bedrock_agent.delete_knowledge_base(knowledgeBaseId=existing_kb_id)

if existing_kb_id:
    start = time.time()
    while True:
        kb_ids = []
        paginator = bedrock_agent.get_paginator('list_knowledge_bases')
        for page in paginator.paginate():
            kb_ids.extend([k['knowledgeBaseId'] for k in page['knowledgeBaseSummaries']])

        if existing_kb_id not in kb_ids:
            print("✓ Existing KB deleted")
            break

        if time.time() - start > 180:
            raise TimeoutError(f"Timed out waiting for KB {existing_kb_id} deletion")

        print("Waiting for KB deletion to complete...")
        time.sleep(10)


Checking for existing Knowledge Base named 'rag-search-kb'...


In [11]:

# ============================================================================
# SECTION 2: Set up Knowledge Base
# ============================================================================
# --- Create OpenSearch Collection ---
try:
    coll = aoss_client.create_collection(name=COLLECTION_NAME, type='VECTORSEARCH')
    coll_arn = coll['createCollectionDetail']['arn']
    print("Waiting for collection to activate...")
except Exception:
    coll_arn = aoss_client.batch_get_collection(names=[COLLECTION_NAME])['collectionDetails'][0]['arn']

while aoss_client.batch_get_collection(names=[COLLECTION_NAME])['collectionDetails'][0]['status'] != 'ACTIVE':
    time.sleep(5)

# --- Create Knowledge Base (retry for eventual consistency) ---
last_err = None
for attempt in range(1, 8):
    try:
        kb_response = bedrock_agent.create_knowledge_base(
            name=KB_NAME,
            roleArn=role_arn,
            knowledgeBaseConfiguration={
                'type': 'VECTOR',
                'vectorKnowledgeBaseConfiguration': {
                    'embeddingModelArn': f'arn:aws:bedrock:{REGION_NAME}::foundation-model/{EMBEDDING_MODEL}'
                }
            },
            storageConfiguration={
                'type': 'OPENSEARCH_SERVERLESS',
                'opensearchServerlessConfiguration': {
                    'collectionArn': coll_arn,
                    'vectorIndexName': INDEX_NAME,
                    'fieldMapping': {
                        'vectorField': 'bedrock-knowledge-base-default-vector',
                        'textField': 'AMAZON_BEDROCK_TEXT_CHUNK',
                        'metadataField': 'AMAZON_BEDROCK_METADATA'
                    }
                }
            }
        )
        kb_id = kb_response['knowledgeBase']['knowledgeBaseId']
        print(f"✓ Knowledge Base Created: {kb_id}")
        break
    except bedrock_agent.exceptions.ValidationException as e:
        last_err = e
        if 'security_exception' in str(e) and attempt < 7:
            wait_seconds = attempt * 15
            print(f"Attempt {attempt}/7 failed due to AOSS security propagation. Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)
            continue
        raise

if 'kb_id' not in locals():
    raise RuntimeError(f"Failed to create knowledge base after retries: {last_err}")


✓ Knowledge Base Created: DFBFV6YBJP


## Ingest documents to the Knowledge Base

In [12]:

# ============================================================================
# SECTION 3: Ingest Source Data
# ============================================================================
# --- Add S3 Data Source ---
s3_config = {
    'bucketArn': f'arn:aws:s3:::{BUCKET_NAME}'
}
if S3_PREFIX:
    s3_config['inclusionPrefixes'] = [S3_PREFIX]

# Use stable naming to allow reruns.
existing_data_source = None
paginator = bedrock_agent.get_paginator('list_data_sources')
for page in paginator.paginate(knowledgeBaseId=kb_id):
    for ds in page['dataSourceSummaries']:
        if ds['name'] == 's3-docs-with-metadata':
            existing_data_source = ds['dataSourceId']
            break

if existing_data_source:
    ds_id = existing_data_source
    print(f"Reusing existing data source: {ds_id}")
else:
    ds_response = bedrock_agent.create_data_source(
        knowledgeBaseId=kb_id,
        name='s3-docs-with-metadata',
        dataSourceConfiguration={
            'type': 'S3',
            's3Configuration': s3_config
        }
    )
    ds_id = ds_response['dataSource']['dataSourceId']
    print(f"Created data source: {ds_id}")

# --- Start Ingestion ---
ingestion_response = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
print(f"✓ Ingestion started for {BUCKET_NAME}/{S3_PREFIX}")
print("You can now filter by 'source_type' or 'doc_index' in your queries!")


Created data source: CYGKJBX224
✓ Ingestion started for rag-search-tests/documents/
You can now filter by 'source_type' or 'doc_index' in your queries!


## Check status of ingestion job

In [13]:
ingestion_job_id = ingestion_response['ingestionJob']['ingestionJobId']
print(f"Monitoring Ingestion Job: {ingestion_job_id}...")

while True:
    job_response = bedrock_agent.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=ingestion_job_id
    )

    status = job_response['ingestionJob']['status']
    print(f"Current Status: {status}")

    if status in ['COMPLETE', 'FAILED', 'STOPPED']:
        break

    time.sleep(10)

# --- Detailed Reporting ---
job_data = job_response['ingestionJob']

print("\n" + "="*30)
print("INGESTION JOB SUMMARY")
print("="*30)
print(f"Status: {job_data['status']}")

# Print Statistics if available
if 'statistics' in job_data:
    stats = job_data['statistics']
    print(f"Documents Scanned: {stats.get('numberOfDocumentsScanned', 0)}")
    print(f"Documents Indexed: {stats.get('numberOfNewDocumentsIndexed', 0)}")
    print(f"Documents Failed:  {stats.get('numberOfDocumentsFailed', 0)}")

# Print Failure Reasons
if 'failureReasons' in job_data:
    print("\nFailure Reasons:")
    for reason in job_data['failureReasons']:
        print(f"  - {reason}")

# Full Debugging Dump
print("\nFull JSON Job Details (for debugging):")
print(json.dumps(job_data, indent=2, default=str))


Monitoring Ingestion Job: OVQBMLLVKU...
Current Status: IN_PROGRESS
Current Status: IN_PROGRESS
Current Status: IN_PROGRESS
Current Status: COMPLETE

INGESTION JOB SUMMARY
Status: COMPLETE
Documents Scanned: 20
Documents Indexed: 20
Documents Failed:  0

Full JSON Job Details (for debugging):
{
  "knowledgeBaseId": "DFBFV6YBJP",
  "dataSourceId": "CYGKJBX224",
  "ingestionJobId": "OVQBMLLVKU",
  "status": "COMPLETE",
  "statistics": {
    "numberOfDocumentsScanned": 20,
    "numberOfMetadataDocumentsScanned": 20,
    "numberOfNewDocumentsIndexed": 20,
    "numberOfModifiedDocumentsIndexed": 0,
    "numberOfMetadataDocumentsModified": 0,
    "numberOfDocumentsDeleted": 0,
    "numberOfDocumentsFailed": 0
  },
  "startedAt": "2026-04-20 17:49:57.502506+00:00",
  "updatedAt": "2026-04-20 17:50:20.498968+00:00"
}
